In [1]:
# Pipeline Configuration
RUN_MODE = "production"  # "test" or "production"
TEST_SAMPLE_SIZE = 10
MAX_WORKERS = 8
CHUNK_SIZE = 1000
MAX_RETRIES = 5
TIMEOUT = 30
ENABLE_CACHE = True
ENABLE_CHECKPOINT = True
SAVE_PREVIEW_IMAGES = True
PIPELINE_VERSION = "1.0.0"


In [2]:
# !pip -q install geopandas rasterio rioxarray pystac-client planetary-computer odc-stac shapely pyproj xarray folium leafmap

In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np

from shapely.geometry import Point

import rasterio
import rioxarray

import planetary_computer
import pystac_client

In [4]:
import os

folders = [
    "../data",
    "../../data/raw",
    "../../data/processed",
    "../../data/features",
    "../../data/metadata",
    "../../data/final",
    "../../data/lucas",
    "../../data/sentinel",
    "../../data/weather",
    "../../data/soilgrids",
    "../outputs",
    "../../outputs/csv",
    "../../outputs/maps",
    "../../outputs/figures",
    "../../outputs/reports",
    "../../outputs/metrics",
    "../../outputs/learning_curves",
    "../../outputs/feature_importance",
    "../../outputs/confusion_matrix",
    "../models",
    "../models/machine_learning",
    "../models/deep_learning",
    "../models/ensemble",
    "../models/best",
    "../models/experimental"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")



Created: ../data
Created: ../../data/raw
Created: ../../data/processed
Created: ../../data/features
Created: ../../data/metadata
Created: ../../data/final
Created: ../../data/lucas
Created: ../../data/sentinel
Created: ../../data/weather
Created: ../../data/soilgrids
Created: ../outputs
Created: ../../outputs/csv
Created: ../../outputs/maps
Created: ../../outputs/figures
Created: ../../outputs/reports
Created: ../../outputs/metrics
Created: ../../outputs/learning_curves
Created: ../../outputs/feature_importance
Created: ../../outputs/confusion_matrix
Created: ../models
Created: ../models/machine_learning
Created: ../models/deep_learning
Created: ../models/ensemble
Created: ../models/best
Created: ../models/experimental


In [5]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected Successfully")

Connected Successfully


In [6]:
# collections = catalog.get_collections()

# for c in collections:
    # print(c.id)
print("Skipping collections listing for speed.")



Skipping collections listing for speed.


In [7]:
from shapely.geometry import Point

lon = 31.083
lat = 30.563

point = Point(lon, lat)

buffer = point.buffer(0.001)

geometry = buffer.__geo_interface__

geometry

{'type': 'Polygon',
 'coordinates': (((31.084, 30.563),
   (31.08399518472667, 30.562901982859668),
   (31.083980785280403, 30.562804909677983),
   (31.08395694033573, 30.562709715322743),
   (31.08392387953251, 30.562617316567632),
   (31.083881921264346, 30.56252860326317),
   (31.0838314696123, 30.56244442976698),
   (31.08377301045336, 30.562365606715836),
   (31.083707106781183, 30.562292893218814),
   (31.08363439328416, 30.562226989546637),
   (31.083555570233017, 30.562168530387698),
   (31.083471396736826, 30.56211807873565),
   (31.083382683432365, 30.562076120467488),
   (31.083290284677254, 30.562043059664266),
   (31.083195090322015, 30.562019214719594),
   (31.08309801714033, 30.562004815273326),
   (31.083, 30.561999999999998),
   (31.082901982859667, 30.562004815273326),
   (31.082804909677982, 30.562019214719594),
   (31.082709715322743, 30.562043059664266),
   (31.08261731656763, 30.562076120467488),
   (31.08252860326317, 30.56211807873565),
   (31.08244442976698, 30

In [8]:
import pandas as pd
import numpy as np
import os
import json
import geopandas as gpd
from shapely.geometry import Point
from sklearn.model_selection import GroupKFold

df_path = "../../data/processed/scaled_features.csv"
if not os.path.exists(df_path):
    df_path = "../../data/processed/merged_features.csv"
if not os.path.exists(df_path):
    print("No compiled features dataset found. Stopping.")
else:
    df = pd.read_csv(df_path)
    # Convert numeric object columns to float to prevent parquet serialization error
    for col in ["OC", "CaCO3", "P", "N", "K"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    # Convert string object columns to native pandas string type
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype("string")
    # Save versioned dataset
    v_str = PIPELINE_VERSION.replace('.', '')
    os.makedirs("../../data/final", exist_ok=True)
    csv_path = f"../../data/final/training_dataset_v{v_str}.csv"
    parquet_path = f"../../data/final/training_dataset_v{v_str}.parquet"
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    # Update latest
    df.to_parquet("../../data/final/latest_training_dataset.parquet", index=False)
    print(f"Versioned final datasets saved successfully: {csv_path}")
    # Spatial Split (GroupKFold by rounded grid bins to prevent spatial leakage)
    df["grid_bin"] = (df["Latitude"].round(1).astype(str) + "_" + df["Longitude"].round(1).astype(str))
    gkf = GroupKFold(n_splits=5)
    # Assign splits
    train_idx, test_idx = next(gkf.split(df, groups=df["grid_bin"]))
    train_df = df.iloc[train_idx]
    test_df = df.iloc[test_idx]
    gkf_val = GroupKFold(n_splits=4)
    train_val_idx, val_idx = next(gkf_val.split(train_df, groups=train_df["grid_bin"]))
    val_df = train_df.iloc[val_idx]
    train_df = train_df.iloc[train_val_idx]
    train_df.to_parquet("../../data/final/train.parquet", index=False)
    val_df.to_parquet("../../data/final/validation.parquet", index=False)
    test_df.to_parquet("../../data/final/test.parquet", index=False)
    print(f"Spatial splits written: Train {len(train_df)}, Val {len(val_df)}, Test {len(test_df)}")
    # Deep Learning json
    records = []
    for _, row in df.iterrows():
        pid = int(row["POINT_ID"])
        lbl = {"Nitrogen": row.get("N", np.nan), "SOC": row.get("OC", np.nan),
               "Clay": row.get("clay", np.nan), "Sand": row.get("sand", np.nan),
               "Silt": row.get("silt", np.nan), "pH": row.get("pH_H2O", np.nan)}
        records.append({
            "POINT_ID": pid,
            "Latitude": float(row["Latitude"]),
            "Longitude": float(row["Longitude"]),
            "Survey_Date": str(row["Survey_Date"]),
            "Image_Path": {
                "s2": f"data/features/image_patches/s2/s2_{pid}.tif",
                "s1": f"data/features/image_patches/s1/s1_{pid}.tif",
                "dem": f"data/features/image_patches/dem/dem_{pid}.tif"
            },
            "Labels": lbl
        })
    with open("../../data/features/dataset.json", "w") as f:
        json.dump(records, f, indent=4)
    # Labels dataset
    os.makedirs("../../data/features/labels", exist_ok=True)
    lbl_cols = ["POINT_ID", "N", "OC", "clay", "sand", "silt", "pH_H2O"]
    lbl_cols = [c for c in lbl_cols if c in df.columns]
    df[lbl_cols].to_csv("../../data/features/labels/labels.csv", index=False)
    # GIS formats
    os.makedirs("../../data/features/gis", exist_ok=True)
    geometry = [Point(xy) for xy in zip(df["Longitude"], df["Latitude"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")
    try:
        gdf.to_file("../../data/features/gis/dataset.geojson", driver="GeoJSON")
        gdf.to_file("../../data/features/gis/dataset.gpkg", driver="GPKG")
        # Shapefile has 10-char col limit, simplify column names to prevent shapefile error
        gdf_shp = gdf.copy()
        gdf_shp.columns = [c[:10] for c in gdf_shp.columns]
        gdf_shp.to_file("../../data/features/gis/dataset.shp")
        print("GIS formats generated successfully in data/features/gis/")
    except Exception as e:
        print(f"GIS export warnings (some drivers might be missing): {e}")



C:\Users\Moaaz\AppData\Local\Temp\ipykernel_37628\6613933.py:15: DtypeWarning: Columns (8,10,12,13,14,15,17,18,31,33,34,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(df_path)


Versioned final datasets saved successfully: ../../data/final/training_dataset_v100.csv


Spatial splits written: Train 11390, Val 3797, Test 3797


GIS export warnings (some drivers might be missing): GeoDataFrame cannot contain duplicated column names.


In [9]:
import pandas as pd
import os

os.makedirs("../../data/final", exist_ok=True)

# Since we only have the single merged reference point features, save it as the final dataset
if os.path.exists("../../data/processed/merged_features.csv"):
    df = pd.read_csv("../../data/processed/merged_features.csv")
    for col in ["OC", "CaCO3", "P", "N", "K"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype("string")
    df.to_csv("../../data/final/training_dataset.csv", index=False)
    df.to_parquet("../../data/final/training_dataset.parquet", index=False)
    print("Saved final training_dataset.csv and training_dataset.parquet")
else:
    print("merged_features.csv not found, cannot build final dataset.")



C:\Users\Moaaz\AppData\Local\Temp\ipykernel_37628\335809765.py:8: DtypeWarning: Columns (8,10,12,13,14,15,17,18,31,33,34,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../data/processed/merged_features.csv")


Saved final training_dataset.csv and training_dataset.parquet
